# Weak- vs. Super-Weak formulation

## Whats new?

This tutorial demonstrates the implementation of the super-weak formulation,


Here, the following topics are covered:
- Implementation of a standard weak formulation for scalar advection 
- Implementation of the same operator as a super-weak formulation

### Note: 
1. This tutorial can be found in the source code repository as as `WeakVsSuperweak.ipynb`. 
   One can directly load this into Jupyter to interactively work with the following code examples.
2. First, we need to load the `BoSSSpad.dll` library in order to load BoSSS-specific functions into Jupyter.
   **In the following line, the reference to `BoSSSpad.dll` is required**. 
   You must either set `#r "BoSSSpad.dll"` to something which is appropirate for your computer
   (e.g. `C:\Program Files (x86)\FDY\BoSSS\bin\Release\net5.0\BoSSSpad.dll` if you installed the binary distribution),
   or, if you are working with the source code, you must compile `BoSSSpad` and put it side-by-side to this worksheet file
   (from the original location in the repository, you can use the scripts `getbossspad.sh`, resp. `getbossspad.bat`).

In [ ]:
//#r "C:\Aktuell_BoSSS_2\experimental\public\src\L4-application\BoSSSpad\bin\Debug\net6.0\BoSSSpad.dll"
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();


## Theoretical Background

### The general case

We assume a generic divergence-form of a spatial derivative operator based on a flux $ \vec{f} $, i.e,
$$
   \text{Op}(u) := \text{div}(\vec{f}(u)) .
$$
We want to solve this operator, i.e., the partial differential equation (PDE) $\text{Op}(u_\text{ex}) = 0$; This is called the strong form.
In discontinuous Galerkin (DG), the approximate solution $u \approx u_\text{ex}$ is the solution of a weak formulation:
$$
  \text{Find } u \text{ so that } a_\text{DG}(u,v) = 0 \text{ for all test functions } v \text{ in the DG space}.
$$

#### The weak formulation

The standard recipe to find the form $a_\text{DG}(u,v)$ 
(Remark: we deviate from standards textbook notation in the respect that in our example here, $a_\text{DG}(u,v)$  form is not necessarily bilinear; 
it might be nonlinear in $u$, but it is linear in $v$.)
for a DG method goes as follows:
1. Multiply the strong form with a test function $v$
2. Integrate over a cell $K_j$
3. Perform partial integration
4. Replace the flux on the boundary integral by a Riemann solver $\hat{F}$
5. Sum over all cells

(More details can be found in any DG textbook.)

Such a derivation yields the form
$$
   a_\text{DG}(u,v) = \sum_{K_j} \left( 
          \oint_{\partial K_j} \hat{F}(u^{\text{in}}, u^{\text{out}}, \vec{n}_{\partial K_j}) v \ \text{dS}  
        - \int_{K_j} \vec{f} \cdot \nabla v \ \text{dV}
        \right) .
$$
This is still a cell-local formulation; one can replace the summation over all cells by an integral over the skeleton of all cell boundaries $\Gamma$ and the entire domain $\Omega$,
to obtain the equivalent formulation 
$$
   a_\text{DG}(u,v) = 
         \oint_{\Gamma} \hat{F}(u^{\text{in}}, u^{\text{out}}, \vec{n}_{\Gamma}) (v^{\text{in}} - v^{\text{out}}) \ \text{dS}  
       - \int_{\Omega} \vec{f} \cdot \nabla v \ \text{dV}.
$$
The latter one is used in BoSSS.

#### The super-weak formulation

The super-weak formulation is obtained by performing a second 
integration-by-parts on the volume term. One obtains:
$$
\begin{aligned}
   a_\text{DG}(u,v) & = \sum_{K_j} \left( 
      \oint_{\partial K_j} (\hat{F}(u^{\text{in}}, u^{\text{out}}, \vec{n}_{\partial K_j}) - \vec{f} \cdot \vec{n}_{\partial K_j} ) v \ \text{dS}  
      + \int_{K_j} \text{div}(\vec{f}) v \ \text{dV}.
      \right) \\
                    & =
      \oint_{\Gamma} \hat{F}(u^{\text{in}}, u^{\text{out}}, \vec{n}_{\Gamma}) (v^{\text{in}} - v^{\text{out}}) 
             - (\vec{f}^{\text{in}} \cdot \vec{n}_{\Gamma} v^{\text{in}} - \vec{f}^{\text{ot}} \cdot \vec{n}_{\Gamma} v^{\text{ot}} ) \ \text{dS}  
      + \int_{\Omega} \text{div}(\vec{f}) v  \ \text{dV}
\end{aligned}
$$

Mathematically, the super-weak formulation is **equivalent** to the weak formulation.
Therefore, for implementation, one is free to chose between the two.
Typically, this depends on which form of the volume term, i.e., $\text{div}(\vec{f}) v$ (super-weak) or $\vec{f} \cdot \nabla v$ (weak),
can be implemented more easily, or more efficiently.

### The super-weak formulation for operators which are **not** in divergence form

One case, where the super-weak form is advantageous is, if the operator is not "fully" in divergence form,
i.e., it could be written as:
$$
   \text{Op}(u) := \text{div}(\vec{f}(u)) + \text{Rest} .
$$
One example would be an advection term for a non-divergence-free velocity field $\text{div}(\vec{V})$, e.g.,
$$
   \text{Op}(u) := \nabla u \cdot \vec{V} = \text{div}(u \vec{V}) - u \text{div}(\vec{V})  .
$$
Using the weak form for the first part, one would have to add the $\text{Rest}$ as an explicit source term:
$$
   a_\text{DG}(u,v) = 
         \oint_{\Gamma} \hat{F}(u^{\text{in}}, u^{\text{out}}, \vec{n}_{\Gamma}) (v^{\text{in}} - v^{\text{out}}) \ \text{dS}  
       - \int_{\Omega} \vec{f} \cdot \nabla v \ \text{dV}
       + \int_{\Omega} \text{Rest} ~ v \ \text{dV}.
$$
This can be inconvenient; E.g., in in the example above, one would have to compute $\text{div}(\vec{V})$.

In the super-weak-form, however, the source term for $\text{Rest}$ can be re-combined with the 
volume term and the original operator formulation can be used:
$$
\begin{aligned}
   a_\text{DG}(u,v) & =
      \oint_{\Gamma} \ldots \ \text{dS}  
      + \int_{\Omega} \text{div}(\vec{f}) v  \ \text{dV}
      + \int_{\Omega} \text{Rest} ~ v \ \text{dV} \\
                    & =
         \oint_{\Gamma} \ldots \ \text{dS}  
        + \int_{\Omega} \ \text{Op}(u) v \ \text{dV}
\end{aligned}
$$

In the case of the above-mentioned convection operator $\nabla u \cdot \vec{V}$, this yields
$$
a_\text{DG}(u,v)  =
    \oint_{\Gamma} \hat{F}(u^{\text{in}}, u^{\text{out}}, \vec{n}_{\Gamma}) (v^{\text{in}} - v^{\text{out}}) 
             - (\vec{V} \cdot \vec{n}_{\Gamma}) (u^{\text{in}} v^{\text{in}} - u^{\text{ot}} v^{\text{ot}} ) \ \text{dS}  
      + \int_{\Omega} \nabla u \cdot \vec{V} v  \ \text{dV} .
$$
This form is still mathematically equivalent, but one can omit the computation of $\text{div}(\vec{V})$,
which seems more elegant.

## Implementation of the respective equation components

In the following, we demonstrate the implementation of:
- the weak form for $\text{div}(u \vec{V})$
- the super-weak form for $\nabla u \cdot \vec{V}$

For a divergence-free field $\vec{V}$, i.e., $\text{div}(\vec{V}) = 0$,
both implementations should give the same result/residual.
We are going to us this to test the implementation.

### Prerequisites

We define a divergence-free velocity field and a standard upwind-flux,
which will be used for both, the weak and the super-weak formulation.

In [ ]:
static Vector VelocityField(Vector X) {
    if(X.Dim != 2)
        throw new ArgumentException("expecting 2D input vector");
    Vector U = new Vector(X.y, -X.x);
    return U;
}

In [ ]:
static double UpwindFlux(Vector X, Vector N, double uIn, double uOt) {
    var Vel = VelocityField(X);
    if(Vel*N > 0) {
        return (Vel*N)*uIn;
    } else {
        return (Vel*N)*uOt;
    }
}

### Implementation of the weak form

For sake of simplicity, the upwind flux is also used at the boundary and the outer value is set to 1.0.
Note: this is for the operator $\text{div}(u \vec{V})$.

In [ ]:
class WeakForm :  IEdgeForm, IVolumeForm {

    public IList<String> ArgumentOrdering {
        get { return new string[] { "u" }; }
    }

    public IList<String> ParameterOrdering => null;

    public TermActivationFlags VolTerms => TermActivationFlags.GradV | TermActivationFlags.UxGradV; 

    public TermActivationFlags InnerEdgeTerms => TermActivationFlags.V | TermActivationFlags.UxV; 

    public TermActivationFlags BoundaryEdgeTerms => TermActivationFlags.V | TermActivationFlags.UxV; 
           
    public double VolumeForm(ref CommonParamsVol cpv, double[] U, double[,] GradU, double V, double[] GradV) {
        double u = U[0];
        var Vel = VelocityField(cpv.Xglobal);
        return -((Vel*u)*GradV);
    }

    public double InnerEdgeForm(ref CommonParams inp, double[] Uin, double[] Uout, double[,] _Grad_uIN, double[,] _Grad_uOUT, double V_IN, double V_OT, double[] _Grad_vIN, double[] _Grad_vOUT) {
        //return 0;
        return UpwindFlux(inp.X, inp.Normal, Uin[0], Uout[0])*(V_IN - V_OT);
    }

    public double BoundaryEdgeForm(ref CommonParamsBnd inp, double[] Uin, double[,] GradUin, double vIn, double[] GradVin) {
        return UpwindFlux(inp.X, inp.Normal, Uin[0], 1.0)*vIn;
    }
}

### Implementation of the super-weak form

For sake of simplicity, the upwind flux is also used at the boundary and the outer value is set to 1.0.
Note: this is for the operator $\nabla u \cdot \vec{V})$.


In [ ]:
class SuperWeakForm :  IEdgeForm, IVolumeForm {

    public IList<String> ArgumentOrdering {
        get { return new string[] { "u" }; }
    }

    public IList<String> ParameterOrdering => null;

    // **different** to the weak form
    public TermActivationFlags VolTerms => TermActivationFlags.V | TermActivationFlags.GradUxV; 

    // same as for the weak form
    public TermActivationFlags InnerEdgeTerms => TermActivationFlags.V | TermActivationFlags.UxV; 

    // same as for the weak form
    public TermActivationFlags BoundaryEdgeTerms => TermActivationFlags.V | TermActivationFlags.UxV; 
           
    public double VolumeForm(ref CommonParamsVol cpv, double[] U, double[,] GradU, double v, double[] GradV) {
        // equal to div(Vel*u) for div(Vel) = 0
        double u = U[0];
        var Vel = VelocityField(cpv.Xglobal);
        var Grad_u = GradU.GetRowPt(0);
        return (Vel*Grad_u)*v;
    }

    public double InnerEdgeForm(ref CommonParams inp, double[] Uin, double[] Uout, double[,] _Grad_uIN, double[,] _Grad_uOUT, double V_IN, double V_OT, double[] _Grad_vIN, double[] _Grad_vOUT) {
        var Vel = VelocityField(inp.X);
        var uIn = Uin[0];
        var uOt = Uout[0];

        //return -(Vel*inp.Normal*uIn*V_IN + Vel*(-1.0*inp.Normal)*uOt*V_OT);
        double Flux = UpwindFlux(inp.X, inp.Normal, Uin[0], Uout[0]);
        return (Flux - (Vel*uIn)*inp.Normal)*V_IN - (Flux - (Vel*uOt)*inp.Normal)*V_OT;
    }

    public double BoundaryEdgeForm(ref CommonParamsBnd inp, double[] Uin, double[,] GradUin, double vIn, double[] GradVin) {
        var Vel = VelocityField(inp.X);
        var uIn = Uin[0];
        
        // V1 ok
        //return 0;
        
        // V2 ok
        return (UpwindFlux(inp.X, inp.Normal, Uin[0], 1.0) - (Vel*uIn)*inp.Normal )*vIn;
    }
}

## Testing

Since $\text{div}(\vec{V}) = 0$, we expect the same result for the weak and the super-weak form.
The result of both forms will be different if $\text{div}(\vec{V}) \neq 0$.

Definition of a 2D grid:

In [ ]:
var grd = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1, 1, 5), GenericBlas.Linspace(-1, 1, 5));

A DG-basis of order 2 on this grid:

In [ ]:
Basis b = new Basis(grd, 2);

We define some non-zero DG field for which the operators will be evaluated:

In [ ]:
SinglePhaseField u0 = new SinglePhaseField(b, "u0");
u0.ProjectField((x, y) => (1-x).Pow2()*(1-y).Pow2());

In [ ]:
u0.CoordinateVector.L2Norm()

In [ ]:
NUnit.Framework.Assert.NotZero(u0.CoordinateVector.L2Norm());

### Evaluation of the weak form

We create an operator and evaluate the residual with respect to `u0`:

In [ ]:
var OperatorWeak = (new WeakForm()).Operator();

In [ ]:
SinglePhaseField WeakFlux = new SinglePhaseField(b);
OperatorWeak.Evaluate(u0, WeakFlux);
WeakFlux.L2Norm()

### Evaluation of the super weak form

We create an operator and evaluate the residual with respect to `u0`:

In [ ]:
var OperatorSuperWeak = (new SuperWeakForm()).Operator();

In [ ]:
SinglePhaseField SuperWeakFlux = new SinglePhaseField(b);
OperatorSuperWeak.Evaluate(u0, SuperWeakFlux);
SuperWeakFlux.L2Norm()

### Comparison and testing

In [ ]:
WeakFlux.CoordinateVector.L2Distance(SuperWeakFlux.CoordinateVector)

In [ ]:
NUnit.Framework.Assert.Less(WeakFlux.CoordinateVector.L2Distance(SuperWeakFlux.CoordinateVector), 1.0e-10, "Residual difference between weak and super weak form.");